# 🛢️ Oil Well Profitability & Risk Analysis

## Project Overview

This project evaluates three oil exploration regions using **Linear Regression**, profitability analysis, and **Bootstrapping**.

The objective is to identify the region that offers the most attractive combination of expected profit and financial risk under the following business constraints:

- 500 candidate wells are studied in each exploration round;
- the 200 wells with the highest predicted reserves are selected;
- the development budget is **$100 million**;
- each unit of `product` represents one thousand barrels;
- each unit generates **$4,500 in revenue**;
- only regions with a probability of loss below **2.5%** should be considered.

The analysis connects machine learning predictions to a practical capital-allocation decision.

## 1. Data Loading and Initial Inspection

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Load data from the three regions

geo_data_0 = pd.read_csv('../data/geo_data_0.csv')
geo_data_1 = pd.read_csv('../data/geo_data_1.csv')
geo_data_2 = pd.read_csv('../data/geo_data_2.csv')

In [16]:
# Preview the first rows

print(geo_data_0.head())
print(geo_data_1.head())
print(geo_data_2.head())

      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.081196 -11.506057  4.999415  137.945408
4  AHL4O  12.702195  -8.147433  5.004363  134.766305
      id        f0        f1        f2     product
0  fwXo0 -1.146987  0.963328 -0.828965   27.758673
1  WJtFt  0.262778  0.269839 -2.530187   56.069697
2  ovLUW  0.194587  0.289035 -5.586433   62.871910
3  q6cA6  2.236060 -0.553760  0.930038  114.572842
4  WPMUX -0.515993  1.716266  5.899011  149.600746


In [17]:
# Inspect dataset structure

print(geo_data_0.info())
print(geo_data_1.info())
print(geo_data_2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column  

In [18]:
# Check missing values

print(geo_data_0.isna().sum())
print(geo_data_1.isna().sum())
print(geo_data_2.isna().sum())

id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64


In [19]:
# Check duplicated rows

print(geo_data_0.duplicated().sum())
print(geo_data_1.duplicated().sum())
print(geo_data_2.duplicated().sum())

0
0
0


In [20]:
# Descriptive statistics

print(geo_data_0.describe())
print(geo_data_1.describe())
print(geo_data_2.describe())

                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248      44.288691
min        -1.408605      -0.848218     -12.088328       0.000000
25%        -0.072580      -0.200881       0.287748      56.497507
50%         0.502360       0.250252       2.515969      91.849972
75%         1.073581       0.700646       4.715088     128.564089
max         2.362331       1.343769      16.003790     185.364347
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        1.141296      -4.796579       2.494541      68.825000
std         8.965932       5.119872       1.703572      45.944423
min       -31.609576     -26.358598      -0.018144       0.000000
25%        -6.298551      -8.267985       1.000021      26.953261
50%       

### Data Quality Summary

The three datasets contain 100,000 observations each and the same five columns: `id`, `f0`, `f1`, `f2`, and `product`.

No missing values or fully duplicated rows were identified. The `id` column is used only as a well identifier and is therefore excluded from model training. The numerical features `f0`, `f1`, and `f2` are used to predict `product`, the reserve volume.

## 2. Model Training and Evaluation

In [21]:
def train_model(data):
    
    # Separate features and target
    features = data.drop(['id', 'product'], axis=1)
    target = data['product']
    
    # Train-validation split (75:25)
    features_train, features_valid, target_train, target_valid = train_test_split(
        features,
        target,
        test_size=0.25,
        random_state=12345
    )
    
    # Train the model
    model = LinearRegression()
    model.fit(features_train, target_train)
    
    # Generate predictions
    predictions_valid = model.predict(features_valid)
    
    # Average predicted reserve volume
    mean_prediction = predictions_valid.mean()
    
    # RMSE
    rmse = mean_squared_error(
        target_valid,
        predictions_valid
    ) ** 0.5
    
    print('Average predicted reserve volume:', mean_prediction)
    print('RMSE:', rmse)
    
    return target_valid.reset_index(drop=True), pd.Series(predictions_valid)


print('Region 0')
target_valid_0, predictions_0 = train_model(geo_data_0)

print('\nRegion 1')
target_valid_1, predictions_1 = train_model(geo_data_1)

print('\nRegion 2')
target_valid_2, predictions_2 = train_model(geo_data_2)

Região 0
Volume médio previsto de reservas: 92.59256778438035
REQM: 37.5794217150813

Região 1
Volume médio previsto de reservas: 68.728546895446
REQM: 0.893099286775617

Região 2
Volume médio previsto de reservas: 94.96504596800489
REQM: 40.02970873393434


### Model Results

The three Linear Regression models show an important trade-off between predicted reserve volume and predictive accuracy:

- **Region 0:** average predicted reserves ≈ **92.59** thousand barrels; RMSE ≈ **37.58**.
- **Region 1:** average predicted reserves ≈ **68.73** thousand barrels; RMSE ≈ **0.89**.
- **Region 2:** average predicted reserves ≈ **94.97** thousand barrels; RMSE ≈ **40.03**.

Region 2 has the highest average predicted reserve volume, while Region 1 has by far the lowest RMSE. These metrics alone are not sufficient for the final decision, so profitability and downside risk are evaluated next.

## 3. Break-Even Analysis

In [22]:
# Prepare variables for profit calculation

budget = 100_000_000
number_of_wells = 200
revenue_per_unit = 4_500

# Average investment per well
budget_per_well = budget / number_of_wells

# Minimum reserve volume per well to break even
minimum_volume = budget_per_well / revenue_per_unit

print('Investment per well:', budget_per_well)
print('Minimum reserve volume per well:', minimum_volume)

# Compare with the actual average reserve volume in each region
print('Volume médio - Region 0:', geo_data_0['product'].mean())
print('Volume médio - Region 1:', geo_data_1['product'].mean())
print('Volume médio - Region 2:', geo_data_2['product'].mean())

Investimento por poço: 500000.0
Volume mínimo necessário por poço: 111.11111111111111
Volume médio - Região 0: 92.50000000000001
Volume médio - Região 1: 68.82500000000002
Volume médio - Região 2: 95.00000000000004


### Break-Even Interpretation

With a $100 million budget distributed across 200 wells, the average investment is **$500,000 per well**. At $4,500 of revenue per unit of product, the break-even reserve volume is approximately **111.11 thousand barrels per well**.

The overall average reserve volume of all three regions is below this threshold. However, the business strategy does not develop randomly selected wells: it evaluates 500 candidates and develops the 200 with the highest predicted reserves. Therefore, profitability must be assessed using the selected wells rather than the overall regional averages.

## 4. Profit Calculation for the Highest-Predicted Wells

In [23]:
# Potential profit by region

def calculate_profit(target, predictions, count):
    predictions_sorted = predictions.sort_values(ascending=False)

    selected_target = target[predictions_sorted.index][:count]

    total_volume = selected_target.sum()
    revenue = total_volume * revenue_per_unit
    profit = revenue - budget

    return total_volume, profit

In [24]:
# Select the 200 wells with the highest predictions

count_wells = 200

top_200_predictions_0 = predictions_0.sort_values(ascending=False).head(count_wells)
top_200_predictions_1 = predictions_1.sort_values(ascending=False).head(count_wells)
top_200_predictions_2 = predictions_2.sort_values(ascending=False).head(count_wells)

In [25]:
# Reserve volume of the selected wells

volume_0, profit_0 = calculate_profit(
    target_valid_0, predictions_0, count_wells
)

volume_1, profit_1 = calculate_profit(
    target_valid_1, predictions_1, count_wells
)

volume_2, profit_2 = calculate_profit(
    target_valid_2, predictions_2, count_wells
)

In [26]:
# Calculate potential profit

print('Region 0')
print('Total reserve volume:', volume_0)
print('Potential profit:', profit_0)

print('\nRegion 1')
print('Total reserve volume:', volume_1)
print('Potential profit:', profit_1)

print('\nRegion 2')
print('Total reserve volume:', volume_2)
print('Potential profit:', profit_2)

Região 0
Volume total: 29601.83565142189
Lucro potencial: 33208260.43139851

Região 1
Volume total: 27589.081548181137
Lucro potencial: 24150866.966815114

Região 2
Volume total: 28245.22214133296
Lucro potencial: 27103499.635998324


### Deterministic Profit Comparison

When the 200 wells with the highest predicted reserves are selected from each validation set:

- **Region 0:** total reserves ≈ **29,601.84** units; potential profit ≈ **$33.21 million**.
- **Region 1:** total reserves ≈ **27,589.08** units; potential profit ≈ **$24.15 million**.
- **Region 2:** total reserves ≈ **28,245.22** units; potential profit ≈ **$27.10 million**.

At this stage, Region 0 has the highest potential profit. This is not yet the final decision because the project requires uncertainty and loss risk to be evaluated through Bootstrapping.

## 5. Risk Analysis with Bootstrapping

In [27]:
# Profit and risk analysis with Bootstrapping

def bootstrap_profit(target, predictions):
    state = np.random.RandomState(12345)
    
    profits = []
    
    for i in range(1000):
        target_subsample = target.sample(
            n=500,
            replace=True,
            random_state=state
        )
        
        predictions_subsample = predictions[target_subsample.index]
        
        volume, profit = calculate_profit(
            target_subsample,
            predictions_subsample,
            200
        )
        
        profits.append(profit)
    
    profits = pd.Series(profits)
    
    mean_profit = profits.mean()
    lower = profits.quantile(0.025)
    upper = profits.quantile(0.975)
    risk = (profits < 0).mean() * 100
    
    return profits, mean_profit, lower, upper, risk

In [28]:
# Profit distributions for the three regions

profits_0, mean_profit_0, lower_0, upper_0, risk_0 = bootstrap_profit(
    target_valid_0,
    predictions_0
)

profits_1, mean_profit_1, lower_1, upper_1, risk_1 = bootstrap_profit(
    target_valid_1,
    predictions_1
)

profits_2, mean_profit_2, lower_2, upper_2, risk_2 = bootstrap_profit(
    target_valid_2,
    predictions_2
)

In [29]:
# Mean profit, 95% interval, and risk of loss

print('Region 0')
print('Mean profit:', mean_profit_0)
print('95% interval:', lower_0, '-', upper_0)
print('Risk of loss:', risk_0, '%')

print('\nRegion 1')
print('Mean profit:', mean_profit_1)
print('95% interval:', lower_1, '-', upper_1)
print('Risk of loss:', risk_1, '%')

print('\nRegion 2')
print('Mean profit:', mean_profit_2)
print('95% interval:', lower_2, '-', upper_2)
print('Risk of loss:', risk_2, '%')

Região 0
Lucro médio: 4259385.269105923
Intervalo de confiança de 95%: -1020900.9483793724 - 9479763.533583675
Risco de prejuízo: 6.0 %

Região 1
Lucro médio: 5152227.734432898
Intervalo de confiança de 95%: 688732.2537050088 - 9315475.912570495
Risco de prejuízo: 1.0 %

Região 2
Lucro médio: 4350083.627827557
Intervalo de confiança de 95%: -1288805.473297878 - 9697069.541802654
Risco de prejuízo: 6.4 %


### Bootstrapping Results

After 1,000 Bootstrap simulations:

- **Region 0:** mean profit ≈ **$4.26 million**; 95% interval ≈ **-$1.02 million to $9.48 million**; loss risk = **6.0%**.
- **Region 1:** mean profit ≈ **$5.15 million**; 95% interval ≈ **$0.69 million to $9.32 million**; loss risk = **1.0%**.
- **Region 2:** mean profit ≈ **$4.35 million**; 95% interval ≈ **-$1.29 million to $9.70 million**; loss risk = **6.4%**.

Only Region 1 satisfies the requirement that the probability of loss must remain below 2.5%. It also has the highest mean Bootstrap profit among the three regions.

An important result is that the decision changes after uncertainty is incorporated: Region 0 had the highest deterministic profit when selecting the best 200 wells from the full validation set, but Region 1 provides the strongest result under the specified exploration process and risk constraint.

## 6. Final Conclusion

The analysis demonstrates that the region with the highest apparent profit is not necessarily the best investment once uncertainty is considered.

Although **Region 0** produced the highest deterministic potential profit, its estimated probability of loss was **6.0%**, above the required 2.5% threshold. Region 2 also failed the risk criterion with a **6.4%** probability of loss.

**Region 1** was the only region that satisfied the business risk constraint, with an estimated loss probability of **1.0%** and an average Bootstrap profit of approximately **$5.15 million**.

Therefore, under the assumptions and constraints defined for this project, **Region 1 is selected for development**.

### Limitations

The datasets are synthetic and the analysis assumes a fixed development budget, constant revenue per unit, equal development costs across wells, and Linear Regression as the required model. The results should therefore be interpreted as a demonstration of a machine-learning and risk-analysis framework rather than a real-world oil exploration recommendation.